# FEDE Embedding Fine-Tuning Pipeline

Run each stage in order. Each cell is independent — you can re-run any stage without repeating earlier ones.

| Stage | What it does |
|---|---|
| 0 | Sanity checks |
| 1a | Generate queries (LLM only, no embedding model) |
| 1b | Assemble training pairs (embedding + positive assignment) |
| 2 | Generate eval queries |
| 3 | Train Round 1 |
| 4 | Mine hard negatives |
| 5 | Train Round 2 |
| 6 | Evaluate |
| 7 | Index corpus into Qdrant |

## Configuration

Edit these before running anything.

In [1]:
# ── How many movies to use for training (rest become eval pool) ──────────────
TRAIN_MOVIES     = 1200
N_EVAL_QUERIES   = 100       # held-out eval queries to generate

# ── Output directories ───────────────────────────────────────────────────────
ROUND1_MODEL_DIR = "fede-embeddinggemma/round1"
ROUND2_MODEL_DIR = "fede-embeddinggemma/round2"

# ── Training overrides (None = use config.py defaults) ───────────────────────
EPOCHS      = None   # e.g. 1 for a quick test
BATCH_SIZE  = 4      # per-device batch; try 2 on 8–12GB if you OOM
LR          = None   # e.g. 1e-5
USE_LORA    = True   # False = full fine-tune (needs more VRAM)
CACHED_MNRL_MINI_BATCH = 8   # CachedMNRL sub-batch; lower if OOM (e.g. 4)

# ── Evaluation ───────────────────────────────────────────────────────────────
EVAL_MOVIES = None   # None = full 1338-movie corpus; set e.g. 500 to save RAM

---
## Stage 0 — Sanity checks

Verify environment, credentials, and data before committing hours of compute.

In [2]:
import logging, os, json
from pathlib import Path

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("pipeline")

# ── Project root ─────────────────────────────────────────────────────────────
import finetuning.config as cfg
log.info("Project root: %s", cfg.PROJECT_ROOT)

# ── Tagged scripts ────────────────────────────────────────────────────────────
tagged_files = list(cfg.TAGGED_SCRIPTS_DIR.glob("*.txt"))
log.info("Tagged scripts found: %d", len(tagged_files))
assert len(tagged_files) > 0, f"No tagged scripts in {cfg.TAGGED_SCRIPTS_DIR}"

# ── Metadata ─────────────────────────────────────────────────────────────────
with open(cfg.METADATA_PATH) as f:
    meta = json.load(f)
log.info("Metadata entries: %d", len(meta))

# ── Gemini API key (used by Batch API for dataset generation) ─────────────────
gemini_key = os.getenv("GEMINI_API_KEY", "")
assert gemini_key, "GEMINI_API_KEY is not set in .env"
log.info("Gemini key: %s…%s", gemini_key[:8], gemini_key[-4:])

# ── HF token (for gated embedding model) ─────────────────────────────────────
hf_token = os.getenv("HF_TOKEN", "")
if hf_token:
    log.info("HF_TOKEN present")
else:
    log.warning("HF_TOKEN not set — fine unless you switched to a gated model")

# ── Existing data ─────────────────────────────────────────────────────────────
raw_path  = cfg.FINETUNING_DATA_DIR / "raw_queries.jsonl"
r1_path   = cfg.FINETUNING_DATA_DIR / "training_pairs_r1.jsonl"
r2_path   = cfg.FINETUNING_DATA_DIR / "training_pairs_r2.jsonl"
eval_path = cfg.FINETUNING_DATA_DIR / "eval_queries.json"
ckpt_path = cfg.FINETUNING_DATA_DIR / "querygen_checkpoint.json"

def _count(p): return sum(1 for _ in open(p)) if p.exists() else 0

print(f"\nCurrent data state:")
print(f"  raw_queries.jsonl       : {_count(raw_path):>6} lines")
print(f"  training_pairs_r1.jsonl : {_count(r1_path):>6} lines")
print(f"  training_pairs_r2.jsonl : {_count(r2_path):>6} lines")
print(f"  eval_queries.json       : {'exists' if eval_path.exists() else 'missing'}")
print(f"  querygen_checkpoint.json: {'exists' if ckpt_path.exists() else 'missing'}")
if ckpt_path.exists():
    ckpt = json.loads(ckpt_path.read_text())
    print(f"    → {len(ckpt.get('processed_movies', []))} movies in checkpoint, status={ckpt.get('status', 'in_progress')}")

print("\n✅ Sanity checks passed")

2026-03-22 22:15:47,336 [INFO] Project root: C:\Users\anzez\University\ML\fede
2026-03-22 22:15:47,341 [INFO] Tagged scripts found: 1338
2026-03-22 22:15:47,346 [INFO] Metadata entries: 1384
2026-03-22 22:15:47,346 [INFO] Gemini key: AIzaSyC1…QQDE
2026-03-22 22:15:47,347 [INFO] HF_TOKEN present



Current data state:
  raw_queries.jsonl       :   6946 lines
  training_pairs_r1.jsonl :   6946 lines
  training_pairs_r2.jsonl :      0 lines
  eval_queries.json       : exists
  querygen_checkpoint.json: exists
    → 1200 movies in checkpoint, status=queries_complete

✅ Sanity checks passed


---
## Stage 1a — Generate queries (LLM only)

Calls the LLM for every movie to produce Type A (synopsis) and Type B (scene summary) queries.
No embedding model is loaded during this stage — only API calls.

- Checkpoints every 50 movies. Safe to interrupt and resume.
- Output: `data/finetuning/raw_queries.jsonl`
- Uses structured output (Pydantic schemas) to guarantee valid JSON from the LLM.

In [8]:
# ── Stage 1a: Generate queries via LLM (no embedding model) ─────────────────
from finetuning.dataset.dataset_builder import DatasetBuilder

builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
raw_queries_path = builder.generate_queries(resume=True)

n_queries = sum(1 for _ in open(raw_queries_path))
print(f"\n✅ Stage 1a complete: {n_queries} raw queries → {raw_queries_path}")
print("You can now safely restart the kernel before running Stage 1b.")

2026-03-22 19:50:38,416 [INFO] Scene corpus built: 1200 movies, 117100 total scenes
2026-03-22 19:50:38,421 [INFO] Checkpoint: 1165 movies
2026-03-22 19:50:38,435 [INFO] Found 15 extra movies in JSONL not in checkpoint — adding to skip set
2026-03-22 19:50:38,437 [INFO] 20 movies to process (1180 already done)
2026-03-22 19:50:38,468 [INFO] [1181/1200] Generating queries for Hannah and Her Sisters
2026-03-22 19:50:43,075 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 19:50:48,743 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 19:50:55,607 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 19:51:02,977 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 19:51:03,068 [INFO] [118


✅ Stage 1a complete: 6946 raw queries → /Users/anze/NLP/fede/data/finetuning/raw_queries.jsonl
You can now safely restart the kernel before running Stage 1b.


In [3]:
import gc

# Free QueryGenerator / LLM client before loading the embedding model.
if "builder" in dir():
    del builder
gc.collect()
print("Stage 1a objects released")

Stage 1a objects released


In [4]:
# ── Stage 1b: Assemble training pairs (embedding + positive assignment) ──────
# Reads raw_queries.jsonl, loads embedding model, finds best positive scene
# per query, samples random negatives, and writes training_pairs_r1.jsonl.
#
# No LLM API calls — purely local compute.
# Safe to interrupt and resume — already-processed movies are skipped.

from finetuning.dataset.dataset_builder import DatasetBuilder

builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
r1_path = builder.assemble_pairs(resume=True)

pairs = sum(1 for _ in open(r1_path))
print(f"\n✅ Stage 1b complete: {pairs} training pairs → {r1_path}")

c:\Users\anzez\University\ML\fede\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-22 20:57:12,081 [INFO] Scene corpus built: 1200 movies, 117100 total scenes
2026-03-22 20:57:12,083 [INFO] Loading embedding model google/embeddinggemma-300m on cuda …
2026-03-22 20:57:12,084 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-03-22 20:57:12,087 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-03-22 20:57:12,510 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-03-22 20:57:12,636 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-22 20:57:12,756 [INFO] HTTP Request: HEAD https://huggingface.co/g

FileExistsError: [WinError 183] Cannot create a file when that file already exists: 'C:\\Users\\anzez\\University\\ML\\fede\\data\\finetuning\\querygen_checkpoint.tmp' -> 'C:\\Users\\anzez\\University\\ML\\fede\\data\\finetuning\\querygen_checkpoint.json'

In [5]:
import gc

# Free embedding model + corpus from Stage 1b before loading training model.
if "builder" in dir():
    del builder
gc.collect()
print("Stage 1b objects released")

Stage 1b objects released


In [ ]:
# (Optional) Clear the checkpoint to force a full re-run of Stage 1a
# import os
# ckpt = cfg.FINETUNING_DATA_DIR / "querygen_checkpoint.json"
# if ckpt.exists():
#     os.remove(ckpt)
#     print("Checkpoint cleared")

In [ ]:
# (Legacy) Run both stages in one call — use Stage 1a + 1b cells above instead.
# from finetuning.dataset.dataset_builder import DatasetBuilder
# builder = DatasetBuilder(max_movies=TRAIN_MOVIES)
# r1_path = builder.build(resume=True)
# pairs = sum(1 for _ in open(r1_path))
# print(f"\n✅ Stage 1 complete: {pairs} pairs → {r1_path}")

---
## Stage 2 — Generate eval queries

Generates ~100 held-out evaluation queries from movies **not** in the training set.

⚠️ Must run **after Stage 1** — reads the checkpoint to know which movies to exclude.

In [7]:
from finetuning.corpus.scene_corpus import load_metadata
from finetuning.dataset.query_generator import load_checkpoint
from finetuning.evaluation.dataset_generator import generate_eval_dataset

ckpt = load_checkpoint()
assert ckpt, "No checkpoint found — run Stage 1 first"
training_ids = set(ckpt.get("processed_movies", []))
log.info("Excluding %d training movies from eval pool", len(training_ids))

metadata = load_metadata()
eval_path = generate_eval_dataset(
    corpus_movie_ids=training_ids,
    metadata=metadata,
    n=N_EVAL_QUERIES,
)

with open(eval_path) as f:
    eval_queries = json.load(f)
print(f"\n✅ Stage 2 complete: {len(eval_queries)} eval queries → {eval_path}")
print("Sample:", eval_queries[0])

2026-03-22 21:25:19,391 [INFO] Excluding 1200 training movies from eval pool
2026-03-22 21:25:19,666 [INFO] Generating eval queries from 173 candidate movies (target: 100)
2026-03-22 21:25:24,962 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:27,826 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:31,396 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:36,323 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:38,631 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions "HTTP/1.1 200 OK"
2026-03-22 21:25:42,140 [INFO] HTTP Request: POST https://generativelanguage.googleapis.com/v1beta/openai/chat/completions 


✅ Stage 2 complete: 100 eval queries → C:\Users\anzez\University\ML\fede\data\finetuning\eval_queries.json
Sample: {'query': 'A district attorney leads an extensive investigation into the assassination of a national president.', 'movie_id': 'jfk', 'movie_title': 'JFK'}


In [ ]:
# (Cleanup already handled by cells above)

---
## Stage 3 — Train Round 1

Fine-tunes the base embedding model on the round-1 dataset using `CachedMultipleNegativesRankingLoss`.
Runs retrieval eval (MRR / Accuracy@k) after each epoch if `eval_queries.json` exists.

💡 **VRAM:** Use small `BATCH_SIZE` (2–4), `USE_LORA = True`, and low `CACHED_MNRL_MINI_BATCH` (4–8) on 8–12GB GPUs. Optionally set `PYTORCH_ALLOC_CONF=expandable_segments:True` before starting Jupyter.

In [3]:
from finetuning.training.model import load_model
from finetuning.training.trainer import build_evaluator, build_trainer, load_training_dataset
from finetuning.corpus.scene_corpus import build_scene_corpus

model      = load_model()  # base model from config
train_data = load_training_dataset(r1_path)
log.info("Training samples: %d", len(train_data))

evaluator = None
if eval_path.exists():
    log.info("Building eval corpus for per-epoch evaluation …")
    eval_corpus = build_scene_corpus(max_movies=EVAL_MOVIES)
    evaluator = build_evaluator(eval_path, eval_corpus)
else:
    log.warning("eval_queries.json not found — training without per-epoch eval")

train_kwargs = {}
if EPOCHS     is not None: train_kwargs["num_epochs"]    = EPOCHS
if BATCH_SIZE is not None: train_kwargs["batch_size"]    = BATCH_SIZE
if LR         is not None: train_kwargs["learning_rate"] = LR

trainer = build_trainer(
    model=model,
    train_dataset=train_data,
    output_dir=ROUND1_MODEL_DIR,
    evaluator=evaluator,
    use_lora=USE_LORA,
    cached_mnrl_mini_batch=CACHED_MNRL_MINI_BATCH,
    **train_kwargs,
)

trainer.train()
model.save(ROUND1_MODEL_DIR)
print(f"\n✅ Stage 3 complete: round-1 model saved to {ROUND1_MODEL_DIR}")

c:\Users\anzez\University\ML\fede\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-22 22:16:02,640 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-03-22 22:16:02,644 [INFO] Use pytorch device_name: cuda:0
2026-03-22 22:16:02,645 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-03-22 22:16:03,260 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-03-22 22:16:03,399 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-22 22:16:03,521 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-22 22:16:03,64

Epoch,Training Loss,Validation Loss,Fede-ir-eval Cosine Accuracy@5,Fede-ir-eval Cosine Accuracy@10,Fede-ir-eval Cosine Accuracy@20,Fede-ir-eval Cosine Precision@5,Fede-ir-eval Cosine Precision@10,Fede-ir-eval Cosine Precision@20,Fede-ir-eval Cosine Recall@5,Fede-ir-eval Cosine Recall@10,Fede-ir-eval Cosine Recall@20,Fede-ir-eval Cosine Ndcg@20,Fede-ir-eval Cosine Mrr@20,Fede-ir-eval Cosine Map@20
1,0.724848,No log,0.050000,0.050000,0.060000,0.010000,0.005000,0.003000,0.050000,0.050000,0.060000,0.033294,0.025167,0.025167
2,0.332092,No log,0.030000,0.060000,0.090000,0.006000,0.006000,0.004500,0.030000,0.060000,0.090000,0.031846,0.015873,0.015873


2026-03-22 23:09:30,591 [INFO] Information Retrieval Evaluation of the model on the fede-ir-eval dataset in epoch 1.0 after 1737 steps:
2026-03-22 23:10:54,260 [INFO] Queries: 100
2026-03-22 23:10:54,260 [INFO] Corpus: 1262

2026-03-22 23:10:54,268 [INFO] Score-Function: cosine
2026-03-22 23:10:54,269 [INFO] Accuracy@5: 5.00%
2026-03-22 23:10:54,269 [INFO] Accuracy@10: 5.00%
2026-03-22 23:10:54,269 [INFO] Accuracy@20: 6.00%
2026-03-22 23:10:54,270 [INFO] Precision@5: 1.00%
2026-03-22 23:10:54,270 [INFO] Precision@10: 0.50%
2026-03-22 23:10:54,270 [INFO] Precision@20: 0.30%
2026-03-22 23:10:54,271 [INFO] Recall@5: 5.00%
2026-03-22 23:10:54,271 [INFO] Recall@10: 5.00%
2026-03-22 23:10:54,271 [INFO] Recall@20: 6.00%
2026-03-22 23:10:54,272 [INFO] MRR@20: 0.0252
2026-03-22 23:10:54,272 [INFO] NDCG@20: 0.0333
2026-03-22 23:10:54,272 [INFO] MAP@20: 0.0252
2026-03-22 23:10:54,276 [INFO] Saving model checkpoint to fede-embeddinggemma/round1\checkpoint-1737
2026-03-22 23:10:54,277 [INFO] Save m


✅ Stage 3 complete: round-1 model saved to fede-embeddinggemma/round1


In [5]:
import gc

# Free training objects before loading the corpus + model for hard-negative mining.
for _var in ("model", "trainer", "train_data", "eval_corpus", "evaluator"):
    if _var in globals():
        del globals()[_var]
gc.collect()
print("🧹 Stage 3 objects released")

🧹 Stage 3 objects released


---
## Stage 4 — Mine hard negatives

Uses the round-1 model to replace random negatives with semantically close (but wrong) scenes.
This makes round-2 training significantly harder and usually gives the biggest quality jump.

⚠️ Peak RAM: loads round-1 model + full scene corpus simultaneously.
Pass `--movies N` equivalent by setting `EVAL_MOVIES` above if you OOM.

In [6]:
from finetuning.dataset.negative_miner import CorpusIndex, mine_hard_negatives

r1_model = load_model(ROUND1_MODEL_DIR)

log.info("Building scene corpus …")
corpus = build_scene_corpus(max_movies=EVAL_MOVIES)

log.info("Building embedding index …")
index = CorpusIndex.build(corpus, r1_model)

log.info("Loading round-1 dataset …")
with open(r1_path) as f:
    r1_rows = [json.loads(line) for line in f]

log.info("Mining hard negatives for %d pairs …", len(r1_rows))
r2_path.parent.mkdir(parents=True, exist_ok=True)
with open(r2_path, "w", encoding="utf-8") as out:
    for i, row in enumerate(r1_rows, 1):
        hard_negs = mine_hard_negatives(
            query=row["anchor"],
            movie_id=row["movie_id"],
            index=index,
            model=r1_model,
            n=cfg.HARD_NEGATIVES_PER_QUERY,
        )
        row["negatives"] = hard_negs
        out.write(json.dumps(row, ensure_ascii=False) + "\n")
        if i % 500 == 0:
            log.info("  %d / %d", i, len(r1_rows))

print(f"\n✅ Stage 4 complete: {len(r1_rows)} pairs with hard negatives → {r2_path}")

2026-03-23 00:00:42,805 [INFO] Loading embedding model: fede-embeddinggemma/round1
2026-03-23 00:00:42,808 [INFO] Use pytorch device_name: cuda:0
2026-03-23 00:00:42,809 [INFO] Load pretrained SentenceTransformer: fede-embeddinggemma/round1
2026-03-23 00:00:42,960 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-23 00:00:43,084 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 4328.43it/s]
Loading weights: 0it [00:00, ?it/s]
Gemma3TextModel LOAD REPORT from: fede-embeddinggemma/round1
Key                                                                     | Status     | 
------------------------------------------------------------------------+------------+-
base_model.model.layers.{0...23}.self_attn.v_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.layers.{0...23}.self_attn.


✅ Stage 4 complete: 6946 pairs with hard negatives → C:\Users\anzez\University\ML\fede\data\finetuning\training_pairs_r2.jsonl


---
## Stage 5 — Train Round 2

Continues fine-tuning from the round-1 checkpoint using the hard-negative dataset.

In [7]:
r2_model   = load_model(ROUND1_MODEL_DIR)  # start from round-1 weights
train_data2 = load_training_dataset(r2_path)
log.info("Round-2 training samples: %d", len(train_data2))

evaluator2 = None
if eval_path.exists():
    eval_corpus2 = build_scene_corpus(max_movies=EVAL_MOVIES)
    evaluator2 = build_evaluator(eval_path, eval_corpus2)

trainer2 = build_trainer(
    model=r2_model,
    train_dataset=train_data2,
    output_dir=ROUND2_MODEL_DIR,
    evaluator=evaluator2,
    use_lora=USE_LORA,
    cached_mnrl_mini_batch=CACHED_MNRL_MINI_BATCH,
    **train_kwargs,
)

trainer2.train()
r2_model.save(ROUND2_MODEL_DIR)
print(f"\n✅ Stage 5 complete: round-2 model saved to {ROUND2_MODEL_DIR}")

2026-03-23 00:24:03,778 [INFO] Loading embedding model: fede-embeddinggemma/round1
2026-03-23 00:24:03,781 [INFO] Use pytorch device_name: cuda:0
2026-03-23 00:24:03,781 [INFO] Load pretrained SentenceTransformer: fede-embeddinggemma/round1
2026-03-23 00:24:03,912 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config.json "HTTP/1.1 200 OK"
2026-03-23 00:24:04,034 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|██████████| 314/314 [00:00<00:00, 2743.93it/s]
Loading weights: 0it [00:00, ?it/s]
Gemma3TextModel LOAD REPORT from: fede-embeddinggemma/round1
Key                                                                     | Status     | 
------------------------------------------------------------------------+------------+-
base_model.model.layers.{0...23}.self_attn.v_proj.lora_A.default.weight | UNEXPECTED | 
base_model.model.layers.{0...23}.self_attn.

Epoch,Training Loss,Validation Loss,Fede-ir-eval Cosine Accuracy@5,Fede-ir-eval Cosine Accuracy@10,Fede-ir-eval Cosine Accuracy@20,Fede-ir-eval Cosine Precision@5,Fede-ir-eval Cosine Precision@10,Fede-ir-eval Cosine Precision@20,Fede-ir-eval Cosine Recall@5,Fede-ir-eval Cosine Recall@10,Fede-ir-eval Cosine Recall@20,Fede-ir-eval Cosine Ndcg@20,Fede-ir-eval Cosine Mrr@20,Fede-ir-eval Cosine Map@20
1,0.000000,No log,0.000000,0.000000,0.010000,0.000000,0.000000,0.000500,0.000000,0.000000,0.010000,0.002314,0.000526,0.000526
2,0.000000,No log,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


2026-03-23 01:14:10,648 [INFO] Information Retrieval Evaluation of the model on the fede-ir-eval dataset in epoch 1.0 after 1737 steps:
2026-03-23 01:15:20,840 [INFO] Queries: 100
2026-03-23 01:15:20,840 [INFO] Corpus: 1262

2026-03-23 01:15:20,844 [INFO] Score-Function: cosine
2026-03-23 01:15:20,845 [INFO] Accuracy@5: 0.00%
2026-03-23 01:15:20,845 [INFO] Accuracy@10: 0.00%
2026-03-23 01:15:20,845 [INFO] Accuracy@20: 1.00%
2026-03-23 01:15:20,845 [INFO] Precision@5: 0.00%
2026-03-23 01:15:20,846 [INFO] Precision@10: 0.00%
2026-03-23 01:15:20,846 [INFO] Precision@20: 0.05%
2026-03-23 01:15:20,846 [INFO] Recall@5: 0.00%
2026-03-23 01:15:20,846 [INFO] Recall@10: 0.00%
2026-03-23 01:15:20,847 [INFO] Recall@20: 1.00%
2026-03-23 01:15:20,847 [INFO] MRR@20: 0.0005
2026-03-23 01:15:20,847 [INFO] NDCG@20: 0.0023
2026-03-23 01:15:20,847 [INFO] MAP@20: 0.0005
2026-03-23 01:15:20,850 [INFO] Saving model checkpoint to fede-embeddinggemma/round2\checkpoint-1737
2026-03-23 01:15:20,851 [INFO] Save m


✅ Stage 5 complete: round-2 model saved to fede-embeddinggemma/round2


In [8]:
import gc

# Free the corpus index + r1_model before loading the training model for Round 2.
# The CorpusIndex holds all 117k scene embeddings in RAM — ~350 MB.
for _var in ("r1_model", "index", "corpus", "r1_rows"):
    if _var in globals():
        del globals()[_var]
gc.collect()
print("🧹 Stage 4 objects released")

🧹 Stage 4 objects released


---
## Stage 6 — Evaluate

Runs the held-out eval queries against the full scene corpus and prints MRR / Accuracy@k.

In [9]:
from finetuning.evaluation.pipeline import run_pipeline
from finetuning.evaluation.semantic_retriever import SemanticRetriever

# Evaluate base model for comparison
models_to_eval = {
    "base"   : None,              # None → loads config.EMBEDDING_MODEL_ID
    "round1" : ROUND1_MODEL_DIR,
    "round2" : ROUND2_MODEL_DIR,
}

results = {}
for name, model_path in models_to_eval.items():
    if model_path is not None and not Path(model_path).exists():
        log.warning("Skipping %s — directory not found", name)
        continue
    log.info("Evaluating %s …", name)
    m = load_model(model_path)
    retriever = SemanticRetriever(m)
    metrics = run_pipeline(retriever, eval_path=eval_path)
    results[name] = metrics["summary"]

# Pretty-print comparison table
print(f"\n{'Model':<10} {'MRR':>6}  {'Acc@5':>6}  {'Acc@10':>7}  {'Acc@20':>7}")
print("-" * 45)
for name, s in results.items():
    print(
        f"{name:<10} {s.get('mrr', 0):>6.3f}  "
        f"{s.get('accuracy_at_5', 0):>6.3f}  "
        f"{s.get('accuracy_at_10', 0):>7.3f}  "
        f"{s.get('accuracy_at_20', 0):>7.3f}"
    )

# Save report
report_path = cfg.FINETUNING_DATA_DIR / "eval_report.json"
report_path.write_text(json.dumps(results, indent=2, ensure_ascii=False))
print(f"\n✅ Stage 6 complete — report saved to {report_path}")

2026-03-23 02:07:02,562 [INFO] Evaluating base …
2026-03-23 02:07:02,563 [INFO] Loading embedding model: google/embeddinggemma-300m
2026-03-23 02:07:02,567 [INFO] Use pytorch device_name: cuda:0
2026-03-23 02:07:02,568 [INFO] Load pretrained SentenceTransformer: google/embeddinggemma-300m
2026-03-23 02:07:02,691 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/modules.json "HTTP/1.1 200 OK"
2026-03-23 02:07:02,812 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-23 02:07:02,944 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-03-23 02:07:03,058 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-300m/resolve/main/README.md "HTTP/1.1 200 OK"
2026-03-23 02:07:03,200 [INFO] HTTP Request: HEAD https://huggingface.co/google/embeddinggemma-

ConnectionError: Failed to connect to Qdrant in server mode: Qdrant connection validation failed: [WinError 10061] No connection could be made because the target machine actively refused it

---
## Stage 7 — Index corpus into Qdrant

Embeds all scenes with the final model and upserts them into Qdrant.

⚠️ Make sure Qdrant is running before executing this cell.

In [ ]:
from finetuning.training.model import encode_documents
from vector_db.indexer import SceneRecord, ScriptIndexer

final_model = load_model(ROUND2_MODEL_DIR)
corpus      = build_scene_corpus()  # full corpus, no max_movies
indexer     = ScriptIndexer()

total_scenes = sum(len(e.scenes) for e in corpus.values())
log.info("Indexing %d movies, %d scenes …", len(corpus), total_scenes)

indexed = 0
for i, (movie_id, entry) in enumerate(corpus.items(), 1):
    scene_texts = [s.text for s in entry.scenes]
    embeddings  = encode_documents(final_model, scene_texts, batch_size=64)

    records = [
        SceneRecord(
            movie_id=scene.movie_id,
            movie_title=scene.movie_title,
            scene_id=scene.scene_id,
            scene_index=scene.scene_index,
            text=scene.text,
            embedding=emb.tolist(),
            scene_title=scene.scene_title,
            character_names=scene.character_names,
        )
        for scene, emb in zip(entry.scenes, embeddings)
    ]

    indexer.index_movie_batch(scenes=records, sentences=[])
    indexed += len(records)

    if i % 100 == 0:
        log.info("  %d / %d movies indexed (%d scenes)", i, len(corpus), indexed)

print(f"\n✅ Stage 7 complete: {len(corpus)} movies, {indexed} scenes indexed")